Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/cleaned/cleaned_sales.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df.shape)

Cohort Assignment

In [ ]:
# Each customer ka first purchase month
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')

# Invoice ka month
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# Cohort Index — kitne months baad wapas aaya
df['CohortIndex'] = (df['InvoiceMonth'] - df['CohortMonth']).apply(lambda x: x.n)

print(df[['CustomerID', 'CohortMonth', 'InvoiceMonth', 'CohortIndex']].head(10))

Cohort Matrix

In [ ]:
# Unique customers per cohort per month
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_data.columns = ['CohortMonth', 'CohortIndex', 'Customers']

# Pivot table
cohort_matrix = cohort_data.pivot_table(
    index='CohortMonth',
    columns='CohortIndex',
    values='Customers'
)

print(cohort_matrix.head())

Retention Matrix

In [ ]:
# Retention % — cohort size se divide karo
cohort_size = cohort_matrix.iloc[:, 0]
retention_matrix = cohort_matrix.divide(cohort_size, axis=0).round(3) * 100

print(retention_matrix.head())

Export

In [ ]:
# Export for Power BI
cohort_matrix.reset_index().to_csv('../exports/cohort_matrix.csv', index=False)
retention_matrix.reset_index().to_csv('../exports/retention_matrix.csv', index=False)

print("✅ Cohort files saved!")

KPI Summary

In [ ]:
kpi = {
    'Total Revenue': df['Revenue'].sum(),
    'Total Orders': df['InvoiceNo'].nunique(),
    'Total Customers': df['CustomerID'].nunique(),
    'AOV': df['Revenue'].sum() / df['InvoiceNo'].nunique(),
    'Top Country': df.groupby('Country')['Revenue'].sum().idxmax()
}

kpi_df = pd.DataFrame([kpi])
kpi_df.to_csv('../exports/kpi_summary.csv', index=False)

print("✅ KPI saved!")
print(kpi_df)